# ViT vs CNN — CIFAR-10 모델 구현

**논문**: An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale  
**arxiv**: https://arxiv.org/abs/2010.11929

이 노트북은 구현 가이드 주석만 포함합니다. 각 셀을 채워 직접 구현하세요.

구현 순서:
1. Patch Embedding
2. Positional Encoding
3. Transformer Encoder Block (MSA + MLP)
4. ViT 전체 모델
5. CNN Baseline
6. 간단한 동작 확인

In [ ]:
# 공통 import
import torch
import torch.nn as nn
from einops import rearrange    # einops의 rearrange는 패치 분할 구현을 간결하게 만들어줍니다.

---
## 1. Patch Embedding

**논문 Section 3.1 — Equation (1)**

```
z_0 = [x_class; x_p¹E; x_p²E; ...; x_pᴺE] + E_pos
  x_p^i ∈ R^(P²·C),  E ∈ R^(P²·C × D)
  N = HW / P²
```

구현 전략 A (Conv2d 활용):  
  `nn.Conv2d(in_channels=C, out_channels=D, kernel_size=P, stride=P)`  
  → 패치 분할과 선형 투영을 한 번에 처리

구현 전략 B (einops + Linear):  
  `rearrange(x, 'b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=P, p2=P)`  
  → 직관적이지만 메모리 사용량 주의

In [ ]:
# ── PatchEmbedding (nn.Module) ────────────────────────────────────────────────
#
# 입력: x  shape = (B, C, H, W)
#
# Step 1. 이미지를 N개의 패치로 분할
#         N = (H / patch_size) * (W / patch_size)
#
# Step 2. 각 패치를 D차원으로 선형 투영 (projection)
#         결과 shape = (B, N, D)
#
# Step 3. [CLS] 토큰을 시퀀스 앞에 prepend
#         cls_token shape = (1, 1, D) → expand → (B, 1, D)
#         결과 shape = (B, N+1, D)
#
# Step 4. Positional Encoding 더하기
#         pos_embedding shape = (1, N+1, D)  — 학습 가능한 파라미터(nn.Parameter)
#         논문은 1D learnable position embedding 사용 (Section 3.1)
#
# 출력: z_0  shape = (B, N+1, D)
pass

---
## 2. Multi-Head Self-Attention (MSA)

**논문 Section 3.1 — Equation (2)**

```
MSA(z) = Concat(head_1, ..., head_k) · W_o
head_i  = Attention(z·W_q^i, z·W_k^i, z·W_v^i)
Attention(Q,K,V) = softmax(Q·Kᵀ / √d_k) · V
  d_k = D / k  (head당 차원)
```

참고: `nn.MultiheadAttention`으로 대체 가능하나,
직접 Q/K/V projection → scaled dot-product → concat 구현 권장.

In [ ]:
# ── MultiHeadSelfAttention (nn.Module) ───────────────────────────────────────
#
# 입력: x  shape = (B, N+1, D)
#
# Step 1. Q, K, V 각각 Linear projection
#         W_q, W_k, W_v: nn.Linear(D, D, bias=False)
#
# Step 2. head별로 분할 (reshape + transpose)
#         (B, N+1, D) → (B, num_heads, N+1, d_k)
#         d_k = D // num_heads
#
# Step 3. Scaled Dot-Product Attention
#         scores = Q @ Kᵀ / sqrt(d_k)      shape = (B, num_heads, N+1, N+1)
#         attn   = softmax(scores, dim=-1)
#         attn   = dropout(attn)            (논문 Appendix B 참조)
#         out    = attn @ V                 shape = (B, num_heads, N+1, d_k)
#
# Step 4. head 결합 + output projection
#         (B, num_heads, N+1, d_k) → (B, N+1, D)
#         W_o: nn.Linear(D, D)
#
# 출력: (B, N+1, D)
pass

---
## 3. Transformer Encoder Block

**논문 Section 3.1 — Equations (2)(3)**

```
z'_l  = MSA(LN(z_{l-1})) + z_{l-1}      # (2) — pre-norm 구조
z_l   = MLP(LN(z'_l))   + z'_l          # (3)

MLP: Linear(D → mlp_dim) → GELU → Dropout → Linear(mlp_dim → D) → Dropout
```

주의: 논문은 **Pre-LayerNorm** 구조 사용 (LN을 sublayer 앞에 적용)

In [ ]:
# ── TransformerEncoderBlock (nn.Module) ──────────────────────────────────────
#
# 구성 요소:
#   - norm1: nn.LayerNorm(D)
#   - attn:  MultiHeadSelfAttention
#   - norm2: nn.LayerNorm(D)
#   - mlp:   nn.Sequential(
#               nn.Linear(D, mlp_dim),
#               nn.GELU(),
#               nn.Dropout(dropout),
#               nn.Linear(mlp_dim, D),
#               nn.Dropout(dropout)
#            )
#
# forward:
#   x = x + attn(norm1(x))    # residual connection
#   x = x + mlp(norm2(x))     # residual connection
pass

---
## 4. ViT 전체 모델

**논문 Figure 1 및 Section 3.1**

```
Input Image
  ↓ PatchEmbedding  (+CLS token +PosEmb)
  ↓ Dropout
  ↓ TransformerEncoderBlock × L
  ↓ LayerNorm
  ↓ CLS token 추출 [:, 0, :]   (pool='cls'인 경우)
  ↓ MLP Head: Linear(D → num_classes)
Output: logits
```

In [ ]:
# ── ViT (nn.Module) ───────────────────────────────────────────────────────────
#
# __init__ 파라미터:
#   image_size, patch_size, num_classes, hidden_dim, num_layers,
#   num_heads, mlp_dim, dropout, emb_dropout, pool='cls'
#
# 구성 요소:
#   - patch_embed: PatchEmbedding
#   - emb_dropout: nn.Dropout
#   - transformer: nn.Sequential(*[TransformerEncoderBlock(...) for _ in range(L)])
#   - norm: nn.LayerNorm(hidden_dim)
#   - head: nn.Linear(hidden_dim, num_classes)
#
# forward:
#   x = patch_embed(x)         # (B, N+1, D)
#   x = emb_dropout(x)
#   x = transformer(x)
#   x = norm(x)
#   cls = x[:, 0]              # pool='cls' 방식
#   logits = head(cls)         # (B, num_classes)
pass

---
## 5. CNN Baseline

간단한 ConvNet 구조 (외부 pretrained 모델 사용 금지).

```
Input (B, 3, 32, 32)
  ↓ Conv Block 1: Conv(3→32, k=3, p=1) → BN → ReLU → MaxPool(2)
  ↓ Conv Block 2: Conv(32→64, k=3, p=1) → BN → ReLU → MaxPool(2)
  ↓ Conv Block 3: Conv(64→128, k=3, p=1) → BN → ReLU → MaxPool(2)
  ↓ Flatten
  ↓ FC(128*4*4 → 256) → ReLU → Dropout
  ↓ FC(256 → 10)
Output: logits
```

채널 수 / block 수는 config.yaml의 `cnn` 섹션에서 읽어와야 합니다.

In [ ]:
# ── ConvBlock (nn.Module) ─────────────────────────────────────────────────────
#
# 단일 Conv Block: Conv2d → BatchNorm2d → ReLU → MaxPool2d
# 재사용 가능하도록 별도 모듈로 분리
pass

In [ ]:
# ── CNNBaseline (nn.Module) ───────────────────────────────────────────────────
#
# __init__ 파라미터:
#   num_classes, num_conv_blocks, base_channels, channel_multiplier,
#   kernel_size, pool_size, fc_hidden_dim, dropout
#
# 구성:
#   - conv_layers: nn.Sequential(*[ConvBlock(...) for i in range(num_conv_blocks)])
#     각 블록의 in/out channel:
#       block i: in=base_channels * channel_multiplier^(i-1), out=base_channels * channel_multiplier^i
#       (block 0: in=3, out=base_channels)
#
#   - classifier: nn.Sequential(
#       nn.Flatten(),
#       nn.Linear(flatten_size, fc_hidden_dim),
#       nn.ReLU(),
#       nn.Dropout(dropout),
#       nn.Linear(fc_hidden_dim, num_classes)
#     )
#
# flatten_size 계산:
#   image_size=32, MaxPool(2) × num_conv_blocks → feature map 크기 = 32 / (2^num_conv_blocks)
#   flatten_size = final_channels * (feature_map_h * feature_map_w)
pass

---
## 6. 동작 확인 (Sanity Check)

구현 후 아래 셀로 shape과 forward pass를 확인하세요.

In [ ]:
# ── Sanity Check ──────────────────────────────────────────────────────────────
#
# 1. 더미 입력 생성
#    x = torch.randn(4, 3, 32, 32)   # batch=4, CIFAR-10
#
# 2. ViT 동작 확인
#    vit = ViT(image_size=32, patch_size=4, num_classes=10, ...)
#    out = vit(x)
#    assert out.shape == (4, 10), f"ViT output shape mismatch: {out.shape}"
#    print(f"ViT params: {sum(p.numel() for p in vit.parameters()):,}")
#
# 3. CNN 동작 확인
#    cnn = CNNBaseline(num_classes=10, ...)
#    out = cnn(x)
#    assert out.shape == (4, 10), f"CNN output shape mismatch: {out.shape}"
#    print(f"CNN params: {sum(p.numel() for p in cnn.parameters()):,}")
pass